# 経済データ分析Ⅱ　第3回
# 重回帰の係数と回帰分析の仮定

前回は住宅データで二つの人口密度を同時に使い、重回帰にすると係数と標準誤差が単回帰から大きく変わることを確かめた。今回は、重回帰の係数がどのように計算され、どのような条件の下で真の係数に近い値になるのかを調べる。

題材は所得と世帯人数から月額消費を説明する仮想データである。データを作る式を決めているので、推定値を真の係数と比べられる。係数の値は日本の家計についての実証結果ではない。

**到達目標**

- 重回帰の係数を、ほかの説明変数で説明できない部分を使った比較として説明する。
- 説明変数を省いた単回帰の係数が重回帰の係数とどう違うかを、数値を使って説明する。
- 回帰分析の標準的な仮定と、各仮定の下で成り立つ性質を述べる。
- 説明変数どうしの相関が強いと係数のばらつきが大きくなる理由を、式から説明する。

## Notebook の使い方

コードセルを上から順に選び、`Shift + Enter` で実行します。途中から始めて変数名のエラーが出たら、先頭のセルから実行し直してください。説明文の記入欄は、ダブルクリックすると編集できます。

ブラウザで使う場合は、このファイルを JupyterLite にアップロードし、**Python (Pyodide)** を選びます。初回のライブラリ読み込みには時間がかかることがあります。ローカルの Jupyter では Python カーネルを使います。各回は単独で実行でき、CSV ファイルの準備は不要です。

コードを書き換えたら必ず再実行してください。画面の出力は、再実行するまで更新されません。授業の終わりには Notebook を保存し、ブラウザで使った場合は `.ipynb` をダウンロードして手元にも残しましょう。

図中の英語は、`Income`＝所得、`Consumption`＝消費、`Household size`＝世帯人数、`Income not explained by household size`＝世帯人数で説明できない所得の部分、`Multiple regression`＝重回帰、`Simple regression`＝単回帰、`Estimated income coefficient`＝所得の係数の推定値です。金額の単位 `10,000 JPY / month` は「月額・万円」を表します。

## 1. 準備

In [ ]:
import sys
if sys.platform == 'emscripten':
    import pyodide_js
    await pyodide_js.loadPackage(['numpy', 'pandas', 'matplotlib', 'statsmodels'])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter
import statsmodels.api as sm
from IPython.display import display

# 表示だけを丸める。計算には元の精度の数値を使う。
def format_table_number(value):
    text = f'{value:.3f}'
    # 表示上だけの負のゼロを避ける。データ自体は変更しない。
    return '0.000' if text == '-0.000' else text

pd.options.display.float_format = format_table_number
plt.rcParams.update({
    'figure.figsize': (8, 4.5), 'figure.dpi': 110,
    'font.family': 'DejaVu Sans', 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.2,
})

# 金額の軸は万円の整数、係数の軸は小数2桁で表示する。
def format_axes(ax, x_money=True, y_money=True):
    ax.xaxis.set_major_formatter(StrMethodFormatter('{x:.0f}' if x_money else '{x:.2f}'))
    ax.yaxis.set_major_formatter(StrMethodFormatter('{x:.0f}' if y_money else '{x:.2f}'))

## 2. 所得・世帯人数・消費の仮想データ

所得が同じでも、世帯人数が多い世帯ほど消費は多いと考えられる。`income`（月額所得、万円）、`household_size`（世帯人数、人）、`consumption`（月額消費、万円）を持つ200世帯の仮想データを作る。

消費は次の式から作る。

$$消費_i=3+0.45\times 所得_i+2\times 世帯人数_i+u_i$$

所得は世帯人数が多いほど高くなるように作る。誤差 $u_i$ は平均0・標準偏差3万円の正規分布から、世帯ごとに独立に、所得や世帯人数とも独立に作る。

In [ ]:
# 説明用の仮想データ。実際の家計調査の数値ではない。
rng = np.random.default_rng(202610)
n = 200
household_size = rng.choice([1, 2, 3, 4, 5], size=n, p=[0.2, 0.3, 0.25, 0.2, 0.05])
income = 18 + 7 * household_size + rng.gamma(2, 4, size=n)
true_error = rng.normal(0, 3, size=n)
consumption = 3 + 0.45 * income + 2 * household_size + true_error

df = pd.DataFrame({'income': income, 'household_size': household_size,
                   'consumption': consumption})
display(df.head())
print(f'所得と世帯人数の相関係数：{df[["income", "household_size"]].corr().iloc[0, 1]:.3f}')

In [ ]:
fig, ax = plt.subplots()
for size in sorted(df['household_size'].unique()):
    group = df[df['household_size'] == size]
    ax.scatter(group['income'], group['consumption'], s=25, alpha=0.65,
               label=f'{size} person(s)')
ax.set(xlabel='Income (10,000 JPY / month)', ylabel='Consumption (10,000 JPY / month)')
ax.legend(title='Household size', fontsize=9)
format_axes(ax)
plt.tight_layout()
plt.show()

世帯人数が多い世帯の点ほど右上にあり、所得と世帯人数の相関係数は約0.83である。

以下では、消費を $y_i$、所得を $x_{1i}$、世帯人数を $x_{2i}$ と書く。$i$ は世帯の番号である。重回帰モデルは第2回と同じ形の

$$y_i=\beta_0+\beta_1x_{1i}+\beta_2x_{2i}+u_i$$

で、このデータの真の係数は $\beta_0=3$、$\beta_1=0.45$、$\beta_2=2$ である。$\beta_1$ は世帯人数が同じときの所得1万円の違いに対応する平均消費の差、$\beta_2$ は所得が同じときの世帯人数1人の違いに対応する平均消費の差を表す。

## 3. 重回帰の最小二乗法

第1回の単回帰と同じく、重回帰でも残差二乗和を最も小さくする係数を選ぶ。三つの係数の候補 $b_0,b_1,b_2$ について

$$S(b_0,b_1,b_2)=\sum_{i=1}^{n}(y_i-b_0-b_1x_{1i}-b_2x_{2i})^2$$

を最小にする組が、OLS 推定値 $\hat\beta_0,\hat\beta_1,\hat\beta_2$ である。三つの係数は同時に決まる。

コードでは `df[['income', 'household_size']]` のように使う説明変数を並べ、`sm.add_constant()` で切片の列を加える。推定値を真の値と並べて表示する。

In [ ]:
X_multiple = sm.add_constant(df[['income', 'household_size']])
multiple = sm.OLS(df['consumption'], X_multiple).fit()
display(pd.DataFrame({'推定値': multiple.params,
                      '真の値': [3, 0.45, 2],
                      '通常の標準誤差': multiple.bse}))

推定値は真の値に近いが、一致はしない。200世帯の誤差 $u_i$ に偶然のばらつきがあるためで、標本が変われば推定値も変わる。その変わり方は第9節で調べる。

### 3.1. 残差が必ず満たす性質

第1回で切片の式を求めたときと同じように、$S$ を各係数で偏微分して0とおくと、残差 $e_i$ について次の三つの式が成り立つ。

$$\sum_{i=1}^{n}e_i=0,\qquad \sum_{i=1}^{n}x_{1i}e_i=0,\qquad \sum_{i=1}^{n}x_{2i}e_i=0$$

残差の合計は0になり、残差は所得とも世帯人数とも相関しない。OLS で係数を求めれば、どのようなデータでも必ず成り立つ性質である。残差 `multiple.resid` を使って確かめる。

In [ ]:
e = multiple.resid
print(f'残差の合計：{e.sum():.1e}')
print(f'所得と残差の積の合計：{(df["income"] * e).sum():.1e}')
print(f'世帯人数と残差の積の合計：{(df["household_size"] * e).sum():.1e}')

表示の `e-12` は $10^{-12}$ を掛けることを表し、例えば `1.7e-12` は $1.7\times10^{-12}$ である。値は計算環境によって少し変わるが、どれも計算機の丸めによる小さな値で、0とみなせる。この性質は第7節で使う。

### 3.2. 係数が一つに決まらない場合

月額所得に加えて、年額所得（月額所得の12倍、万円）を説明変数に入れたとする。

$$b_1\times 月額所得_i+b_2\times 年額所得_i=(b_1+12b_2)\times 月額所得_i$$

なので、データから決まるのは $b_1+12b_2$ という組み合わせだけである。例えば $(b_1,b_2)=(0.60,0)$ と $(0,0.05)$ は同じ当てはめ値を与え、残差二乗和も等しい。$S$ を最小にする係数の組が無数にあるため、係数は一つに決まらない。第2回の $2x+3x=1x+4x$ と同じ状況で、これを**完全な多重共線性**という。

In [ ]:
df['annual_income'] = 12 * df['income']
X_exact = sm.add_constant(df[['income', 'annual_income']])
print(f'説明変数の列数（切片を含む）：{X_exact.shape[1]}')
print(f'独立な列の数（ランク）：{np.linalg.matrix_rank(X_exact)}')

prediction_a = 0.60 * df['income'] + 0.00 * df['annual_income']
prediction_b = 0.00 * df['income'] + 0.05 * df['annual_income']
print('2組の係数の当てはめ値は一致するか：', np.allclose(prediction_a, prediction_b))

**ランク**は、ほかの列の足し算や定数倍で表せない列の本数である。この例では列数3に対してランクが2なので、月額所得か年額所得のどちらか一方を使えば十分である。ソフトウェアによっては完全な多重共線性があっても係数を表示するが、表示された係数は無数にある組のうちの一つである。

## 4. 世帯人数を一定にした比較の計算

第2回では、重回帰の係数を「ほかの説明変数を同じ値に保ったときの差」と説明した。重回帰の所得の係数は、次の2段階の計算でも求められる。

1. 所得を世帯人数に単回帰し、残差を求める。この残差は、所得のうち世帯人数では説明できない部分である。
2. 消費をこの残差に単回帰する。その傾きは、重回帰の所得の係数と一致する。

最初のコードでは、手順1の残差を `income_resid` 列に保存する。

In [ ]:
X_aux = sm.add_constant(df[['household_size']])
aux = sm.OLS(df['income'], X_aux).fit()
df['income_resid'] = aux.resid
display(df[['income', 'household_size', 'income_resid']].head())

先頭の世帯は所得30.7万円の1人世帯である。手順1の回帰では1人世帯の所得は約33.8万円と予測されるので、この世帯の `income_resid` は約−3.2万円になる。

次に、消費を `income_resid` に単回帰し、傾きを重回帰の所得の係数と比べる。

In [ ]:
X_resid = sm.add_constant(df[['income_resid']])
two_step = sm.OLS(df['consumption'], X_resid).fit()
print(f'手順2の傾き：{two_step.params["income_resid"]:.4f}')
print(f'重回帰の所得の係数：{multiple.params["income"]:.4f}')

In [ ]:
fig, ax = plt.subplots()
for size in sorted(df['household_size'].unique()):
    group = df[df['household_size'] == size]
    ax.scatter(group['income_resid'], group['consumption'], s=25, alpha=0.65,
               label=f'{size} person(s)')
grid = np.linspace(df['income_resid'].min(), df['income_resid'].max(), 100)
ax.plot(grid, two_step.params['const'] + two_step.params['income_resid'] * grid,
        color='black', label='Slope = income coefficient')
ax.set(xlabel='Income not explained by household size (10,000 JPY / month)',
       ylabel='Consumption (10,000 JPY / month)')
ax.legend(title='Household size', fontsize=9)
format_axes(ax)
plt.tight_layout()
plt.show()

二つの値は一致する。図の横軸は世帯人数から予測される所得との差なので、同じ色（同じ世帯人数）の世帯どうしの所得の違いに近い。重回帰の所得の係数は、所得の変動のうち世帯人数と重ならないこの部分だけを使って計算されている。所得と世帯人数の相関が強いほど、この部分は小さくなる（第8節）。この結果は Frisch–Waugh–Lovell の定理として知られている。

## 5. 世帯人数を省いた単回帰との比較

**実行前の予想**：世帯人数を使わずに消費を所得だけに回帰すると、所得の係数は重回帰の係数より大きくなるでしょうか、小さくなるでしょうか。所得と世帯人数の関係、世帯人数と消費の関係から考えてください。

### 記入欄：単回帰の係数の予想

ここをダブルクリックし、考えたことを記入してください。`Shift + Enter` で表示に戻ります。

-

消費を所得だけに単回帰した傾きと、重回帰の係数を比べる。二つの間には、どのようなデータでも次の関係が成り立つ。

> 単回帰の傾き ＝ 重回帰の所得の係数 ＋ 重回帰の世帯人数の係数 × 世帯人数を所得に回帰した傾き

最後の「世帯人数を所得に回帰した傾き」は、所得が1万円高い世帯で世帯人数が平均的に何人多いかを表す。

In [ ]:
X_simple = sm.add_constant(df[['income']])
simple = sm.OLS(df['consumption'], X_simple).fit()
size_on_income = sm.OLS(df['household_size'], X_simple).fit().params['income']

display(pd.Series({
    '単回帰の傾き': simple.params['income'],
    '重回帰の所得の係数': multiple.params['income'],
    '重回帰の世帯人数の係数': multiple.params['household_size'],
    '世帯人数を所得に回帰した傾き': size_on_income,
    '右辺の計算結果': multiple.params['income']
                     + multiple.params['household_size'] * size_on_income,
}, name='値').to_frame())

数値を入れると $0.631=0.506+1.257\times0.099$ である。単回帰の傾きには、所得の係数0.506に加えて、所得の高い世帯ほど世帯人数が多いことによる消費の差 $1.257\times0.099\approx0.125$ が含まれる。単回帰は世帯人数の違いも含めた比較、重回帰は世帯人数を同じにした比較であり、係数の違いは比較の条件の違いから生じる。

### 演習1：単回帰と重回帰の係数

1. 所得の係数は、単回帰と重回帰でどう変わりましたか。上の関係式の数値を使って説明してください。
2. 所得が同じで世帯人数が1人多い場合、消費は平均的に何万円多いと推定されますか。
3. 「ほかの条件が同じ」という表現を使うとき、この回帰で同じとしている変数は何でしょうか。年齢や資産も自動的に同じになるでしょうか。

### 記入欄：演習1

ここをダブルクリックし、考えたことを記入してください。`Shift + Enter` で表示に戻ります。

-

<details>
<summary>考えたあとに開く：ヒント・確認</summary>

1. 所得の高い世帯は世帯人数も多く、所得が同じなら世帯人数が多いほど消費も多い。単回帰の傾きには、この世帯人数の分が加わる。
2. 重回帰の世帯人数の係数を読む。「所得が同じ」という条件をつけて答える。
3. 重回帰が一定にしているのは、回帰式に入れた説明変数である。この例では世帯人数だけで、年齢や資産まで調整したことにはならない。

</details>

## 6. 回帰分析の標準的な仮定

第3〜5節の関係は、データを入れれば必ず成り立つ計算上の関係である。推定値が真の係数に近いかどうかは、データがどのように生まれたかによる。そこで次の仮定を置く。

- 仮定1（母数について線形）：母集団で $y_i=\beta_0+\beta_1x_{1i}+\beta_2x_{2i}+u_i$ が成り立つ。係数について一次式であればよく、所得の二乗や対数を説明変数に使ってもよい。
- 仮定2（無作為抽出）：各世帯は、同じ母集団から互いに独立に選ばれる。
- 仮定3（完全な多重共線性がない）：どの説明変数も、ほかの説明変数の足し算や定数倍では完全には表せない。
- 仮定4（誤差の条件付き平均が0）：所得や世帯人数がどの値でも、誤差 $u_i$ の平均は0である。式では $E[u_i\mid x_{1i},x_{2i}]=0$ と書く。
- 仮定5（均一分散）：所得や世帯人数がどの値でも、誤差の分散は同じ値 $\sigma^2$ である。
- 仮定6（正規性）：所得や世帯人数の値によらず、誤差は平均0・分散 $\sigma^2$ の正規分布に従う。

仮定を加えるごとに、OLS 推定量について言えることが増える。

- 仮定1〜4の下で、OLS 推定量は**不偏**である。標本を取り直して推定を繰り返すと、推定値の平均は真の係数に一致する。
- 仮定1〜5の下で、係数の分散は第8節の式で表され、statsmodels が表示する通常の標準誤差を使える。この条件の下では、OLS 推定量は線形で不偏な推定量の中で分散が最も小さい（ガウス・マルコフの定理）。
- 仮定1〜6の下で、標本が小さくても $t$ 検定と信頼区間を正確に計算できる。仮定6が成り立たなくても、標本が大きければ近似として使える。

仮定3は、第3.2節のように係数が一つに決まらない事態を除く条件である。不偏性には仮定5と仮定6は使わない。

今回のデータは仮定1〜6をすべて満たすように作っている。実データで仮定4が成り立つかは、データの成り立ちや経済的な関係から判断する。次の節で見るように、残差からは確かめられない。

## 7. 誤差項と残差

- **誤差項** $u_i$：母集団の式からのずれ。真の係数が分からないため、実データでは計算できない。
- **残差** $e_i$：推定した式からのずれ。観測値と推定結果から計算できる。

今回は真の式を決めた数値実験なので、両方を並べて比べられる。

In [ ]:
error_comparison = df[['income', 'household_size', 'consumption']].head(6).copy()
error_comparison['誤差項'] = true_error[:6]
error_comparison['残差'] = multiple.resid.iloc[:6].to_numpy()
display(error_comparison)
print(f'誤差項の平均：{format_table_number(true_error.mean())}')
print(f'残差の平均：{format_table_number(multiple.resid.mean())}')

平均0の分布から作った誤差でも、200個の平均はちょうど0にはならない。残差の平均は、第3.1節の性質から0になる。

### 7.1. 残差から仮定4を確かめられない理由

世帯人数を省き、消費を所得だけで説明するモデルを考える。このモデルでは、世帯人数による消費の差が誤差に入る。

$$y_i=\beta_0+\beta_1x_{1i}+v_i,\qquad v_i=\beta_2x_{2i}+u_i$$

世帯人数は所得と相関するので、誤差 $v_i$ も所得と相関し、仮定4が成り立たない。一方、所得だけの回帰の残差は第3.1節の性質を満たすので、所得との相関係数は必ず0になる。

In [ ]:
error_v = 2 * df['household_size'] + true_error   # 所得だけの式の誤差 v
display(pd.DataFrame({'所得との相関係数': [
    np.corrcoef(df['income'], error_v)[0, 1],
    np.corrcoef(df['income'], simple.resid)[0, 1],
]}, index=['所得だけの式の誤差 v', '所得だけの回帰の残差']))

誤差と所得の相関係数は約0.47、残差と所得の相関係数は0である。残差が説明変数と相関しないのは OLS の計算の結果なので、仮定4が成り立つ根拠にはならない。

**確認**：第2回の住宅データで残差を計算できても、誤差の条件付き平均が0と確認したことにはならないのはなぜでしょうか。観測されたものと仮定したものを分けて説明してください。

### 記入欄：誤差と残差

ここをダブルクリックし、考えたことを記入してください。`Shift + Enter` で表示に戻ります。

-

## 8. 係数のばらつきと多重共線性

仮定1〜5の下で、重回帰の所得の係数の分散は次の式で表される。導き方は省略し、式が表す内容を読む。

$$\mathrm{Var}(\hat\beta_1)=\frac{\sigma^2}{SST_1\times(1-R_1^2)}$$

- $\sigma^2$：誤差の分散
- $SST_1=\sum_{i=1}^{n}(x_{1i}-\bar x_1)^2$：所得の偏差の二乗和。所得のばらつきが大きいほど、世帯数が多いほど大きい
- $R_1^2$：所得を世帯人数に回帰したときの決定係数。第4節の手順1の回帰である

分母の $SST_1\times(1-R_1^2)$ は第4節の `income_resid` の二乗和に等しく、世帯人数と重ならない所得の変動の大きさを表す。係数のばらつきが大きくなるのは、誤差の分散が大きいとき、所得のばらつきが小さいとき、所得が世帯人数でよく説明されて $R_1^2$ が1に近いときである。最後の場合が多重共線性にあたる。$R_1^2=1$ では分母が0になり、係数は一つに決まらない（第3.2節）。

**分散拡大係数**（Variance Inflation Factor, VIF）は $VIF_1=1/(1-R_1^2)$ で、所得が世帯人数と相関しない場合に比べて分散が何倍になるかを表す。標準誤差では $\sqrt{VIF_1}$ 倍になる。説明変数が二つのとき、$R_1^2$ は二つの説明変数の相関係数の二乗に等しい。

In [ ]:
sst1 = ((df['income'] - df['income'].mean())**2).sum()
r2_1 = aux.rsquared
vif_income = 1 / (1 - r2_1)
print(f'所得を世帯人数に回帰した決定係数 R1^2：{r2_1:.3f}')
print(f'所得の VIF：{vif_income:.2f}（標準誤差では {np.sqrt(vif_income):.2f} 倍）')

実際の分析では $\sigma^2$ が分からないので、残差の二乗和を $n-3$ で割った値で推定する（3は推定した係数の数）。statsmodels では `multiple.scale` がこの推定値である。式の $\sigma^2$ をこの推定値に置き換えて平方根をとると、statsmodels が表示する標準誤差になる。

In [ ]:
sigma_hat = np.sqrt(multiple.scale)
se_formula = sigma_hat / np.sqrt(sst1 * (1 - r2_1))
print(f'式から計算した標準誤差：{se_formula:.4f}')
print(f'statsmodels の標準誤差：{multiple.bse["income"]:.4f}')

所得の VIF は約3.2で、所得が世帯人数と相関しない場合に比べ、所得の係数の分散は約3.2倍、標準誤差は約1.8倍になっている。

VIF が大きいという理由で変数を削除すると、その変数の係数が0でない限り、第5節の単回帰のように残した変数の係数が省いた変数の分を含む。まず変数の定義・単位・重複を調べ、知りたい係数を明確にする。データを増やして $SST_1$ を大きくすることや、説明変数の組み合わせを見直すことも考えられる。所得を万円から円に変えても、$R_1^2$ は変わらない。

## 9. 標本を取り直したときの推定値の分布

所得と世帯人数を固定し、誤差 $u_i$ だけを1,000回作り直す。標本ごとに、重回帰の所得の係数と、所得だけの単回帰の傾きを計算する。

重回帰の係数は、仮定1〜4が成り立つので平均0.45の周りに分布すると考えられる。単回帰の傾きについては、第5節の関係式の重回帰の係数を真の係数（0.45と2）に置き換えると、平均は $0.45+2\times0.099\approx0.65$ になると予想できる。このように、必要な説明変数を省いたことで生じる系統的なずれ（ここでは $2\times0.099$）を**欠落変数バイアス**という。

**実行前の予想**：重回帰と単回帰の所得の係数は、それぞれどの値の周りに分布するでしょうか。どちらの分布が広いでしょうか。

`np.linalg.pinv(X)` は、$y$ の値を並べたものに掛けると OLS の係数が得られる行列を返す。説明変数は固定しているので、この行列を一度だけ計算し、1,000回の推定に使う。

In [ ]:
rng_repeat = np.random.default_rng(202611)
B = 1000
pinv_multiple = np.linalg.pinv(X_multiple.to_numpy())
pinv_simple = np.linalg.pinv(X_simple.to_numpy())
mean_consumption = 3 + 0.45 * df['income'].to_numpy() + 2 * df['household_size'].to_numpy()

slopes_multiple, slopes_simple = [], []
for _ in range(B):
    y_repeat = mean_consumption + rng_repeat.normal(0, 3, n)
    slopes_multiple.append((pinv_multiple @ y_repeat)[1])
    slopes_simple.append((pinv_simple @ y_repeat)[1])

display(pd.DataFrame({
    '推定値の平均': [np.mean(slopes_multiple), np.mean(slopes_simple)],
    '予想した平均': [0.45, 0.45 + 2 * size_on_income],
    '推定値の標準偏差': [np.std(slopes_multiple, ddof=1), np.std(slopes_simple, ddof=1)],
}, index=['重回帰の所得の係数', '単回帰の傾き']))

In [ ]:
fig, ax = plt.subplots()
bins = np.linspace(min(slopes_multiple + slopes_simple), max(slopes_multiple + slopes_simple), 40)
ax.hist(slopes_multiple, bins=bins, alpha=0.6, label='Multiple regression')
ax.hist(slopes_simple, bins=bins, alpha=0.6, label='Simple regression (household size omitted)')
ax.axvline(0.45, color='#bc3b45', linestyle='--', label='True income coefficient')
ax.set(xlabel='Estimated income coefficient', ylabel='Number of samples')
format_axes(ax, x_money=False)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

重回帰の推定値は0.45の周りに、単回帰の推定値は約0.65の周りに分布する。単回帰の分布は重回帰より狭いが、中心は真の係数から離れている。分散の小ささだけでは、推定方法の良し悪しを判断できない。

1,000回の平均や標準偏差も乱数によって変動する。この実験は、仮定の下で成り立つ性質を数値で確かめたものである。

## 10. 演習2：式と仮定から考える

1. 月額所得と、その12倍で作った年額所得を同時に回帰式に入れた。どの仮定が成り立たないか、どう直せばよいかを説明してください。
2. 第2回の住宅データでは、二つの対数人口密度の相関係数が約0.954だった。総面積密度の係数の通常の標準誤差は、総面積密度だけの単回帰で1.85、二つの密度の重回帰で6.12である。VIF とその平方根を計算し、標準誤差の比 $6.12/1.85$ と比べてください。
3. 所得だけの回帰の残差と所得の相関係数が0だったので、「誤差の条件付き平均が0という仮定を確認した」と報告した。この報告の問題点を説明してください。

In [ ]:
# 演習2の2の計算に使ってください。
r_density = 0.954

### 記入欄：演習2

ここをダブルクリックし、考えたことを記入してください。`Shift + Enter` で表示に戻ります。

-

<details>
<summary>考えたあとに開く：ヒント・確認</summary>

1. 年額所得は月額所得の12倍なので、二つの係数を一つに決められない。仮定3が成り立たない例である。どちらか一方だけを使う。
2. $VIF=1/(1-0.954^2)\approx11.1$、その平方根は約3.3で、標準誤差の比 $6.12/1.85\approx3.3$ とほぼ等しい。残る小さな差は、説明変数を加えたことで誤差の分散の推定値が少し変わったことと、相関係数の丸めによる。
3. 残差と説明変数が相関しないことは、第3.1節の性質から必ず成り立つ。仮定4は観測できない誤差についての仮定で、第7.1節の例のように、仮定4が成り立たないモデルでも残差と説明変数の相関は0になる。

</details>

## 関数の資料

- [statsmodels：OLS](https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.OLS.html)
- [NumPy：numpy.linalg.pinv](https://numpy.org/doc/stable/reference/generated/numpy.linalg.pinv.html)
- [statsmodels：VIF](https://www.statsmodels.org/stable/generated/statsmodels.stats.outliers_influence.variance_inflation_factor.html)